In [1]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import xgboost as xgb

# Create models directory if not exists
os.makedirs("models", exist_ok=True)


In [2]:
# Replace with your actual paths
train_df = pd.read_csv("malware_train_by_hash.csv")
test_df  = pd.read_csv("malware_test_by_hash.csv")
full_df  = pd.read_csv("malware.csv")  # optional for reference

# Drop 'hash' column if exists
for df in [train_df, test_df]:
    if "hash" in df.columns:
        df.drop(columns=["hash"], inplace=True)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)


Train shape: (80000, 24)
Test shape: (20000, 24)


In [3]:
# Find non-numeric columns
cat_cols = train_df.select_dtypes(include='object').columns.tolist()
cat_cols = [c for c in cat_cols if c != "label"]  # exclude label

# Encode categorical columns
label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    train_df[col] = le.fit_transform(train_df[col])
    test_df[col]  = le.transform(test_df[col])
    label_encoders[col] = le

# Save label encoders
joblib.dump(label_encoders, "models/malware_label_encoders.joblib")
print("Encoded columns:", cat_cols)


Encoded columns: ['classification']


In [4]:
X_train = train_df.drop(columns=["label"])
y_train = train_df["label"]
X_test  = test_df.drop(columns=["label"])
y_test  = test_df["label"]


In [5]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Save scaler
joblib.dump(scaler, "models/malware_scaler.joblib")


['models/malware_scaler.joblib']

In [6]:
xgb_model = xgb.XGBClassifier(
    n_estimators=250,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42
)

xgb_model.fit(
    X_train_scaled, y_train,
    eval_set=[(X_test_scaled, y_test)],
    verbose=True
)

# Predictions
y_pred = xgb_model.predict(X_test_scaled)

# Metrics
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

# Save model
xgb_model.save_model("models/malware_xgb.json")


[0]	validation_0-logloss:0.65443
[1]	validation_0-logloss:0.60930
[2]	validation_0-logloss:0.56832
[3]	validation_0-logloss:0.53096
[4]	validation_0-logloss:0.50790
[5]	validation_0-logloss:0.47583
[6]	validation_0-logloss:0.45460
[7]	validation_0-logloss:0.42681
[8]	validation_0-logloss:0.40109
[9]	validation_0-logloss:0.37725
[10]	validation_0-logloss:0.35509
[11]	validation_0-logloss:0.33688
[12]	validation_0-logloss:0.31754
[13]	validation_0-logloss:0.29948
[14]	validation_0-logloss:0.28484
[15]	validation_0-logloss:0.26894
[16]	validation_0-logloss:0.25635
[17]	validation_0-logloss:0.24434
[18]	validation_0-logloss:0.23100
[19]	validation_0-logloss:0.21847
[20]	validation_0-logloss:0.20669
[21]	validation_0-logloss:0.19560
[22]	validation_0-logloss:0.18517
[23]	validation_0-logloss:0.17715
[24]	validation_0-logloss:0.16780
[25]	validation_0-logloss:0.15898
[26]	validation_0-logloss:0.15066
[27]	validation_0-logloss:0.14281
[28]	validation_0-logloss:0.13539
[29]	validation_0-loglos

C:\Users\syeds\AppData\Roaming\Python\Python312\site-packages\xgboost\core.py:158: UserWarning: [05:22:52] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0c55ff5f71b100e98-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


[34]	validation_0-logloss:0.09964
[35]	validation_0-logloss:0.09661
[36]	validation_0-logloss:0.09171
[37]	validation_0-logloss:0.08808
[38]	validation_0-logloss:0.08363
[39]	validation_0-logloss:0.07942
[40]	validation_0-logloss:0.07626
[41]	validation_0-logloss:0.07243
[42]	validation_0-logloss:0.06880
[43]	validation_0-logloss:0.06536
[44]	validation_0-logloss:0.06209
[45]	validation_0-logloss:0.05899
[46]	validation_0-logloss:0.05605
[47]	validation_0-logloss:0.05413
[48]	validation_0-logloss:0.05144
[49]	validation_0-logloss:0.04931
[50]	validation_0-logloss:0.04687
[51]	validation_0-logloss:0.04454
[52]	validation_0-logloss:0.04234
[53]	validation_0-logloss:0.04024
[54]	validation_0-logloss:0.03826
[55]	validation_0-logloss:0.03637
[56]	validation_0-logloss:0.03457
[57]	validation_0-logloss:0.03287
[58]	validation_0-logloss:0.03125
[59]	validation_0-logloss:0.02971
[60]	validation_0-logloss:0.02869
[61]	validation_0-logloss:0.02727
[62]	validation_0-logloss:0.02653
[63]	validatio

In [8]:
importance = xgb_model.get_booster().get_score(importance_type='gain')
sorted_importance = sorted(importance.items(), key=lambda x: x[1], reverse=True)
print("Top feature importance (gain):")
for f, v in sorted_importance[:20]:
    print(f"{f}: {v:.4f}")

# Save summary
summary = {"feature_importance_gain": sorted_importance}
import json
with open("models/malware_artifacts_summary.json", "w") as f:
    json.dump(summary, f, indent=4)


Top feature importance (gain):
f1: 4139.5146
f18: 1982.7852
f6: 1195.6456
f20: 1113.7764
f10: 436.9493
f11: 328.5893
f14: 323.1583
f4: 309.2712
f15: 256.3864
f8: 256.0600
f13: 239.3685
f12: 193.2098
f19: 165.0553
f5: 87.0939
f3: 78.9802
f7: 74.0028
f16: 42.3105
f22: 25.4795
f0: 14.2770
f17: 8.1921


In [9]:
# Defender: block if predicted malware with prob > 0.8
probs = xgb_model.predict_proba(X_test_scaled)
pred_label = y_pred
pred_prob  = probs.max(axis=1)

def defender_action(prob, label, block_threshold=0.8):
    if label == 1 and prob >= block_threshold:
        return "block"
    return "allow"

defender_results = pd.DataFrame({
    "predicted_label": pred_label,
    "prediction_prob": pred_prob,
    "defense_action": [defender_action(p,l) for p,l in zip(pred_prob, pred_label)]
})

defender_results.to_csv("models/malware_defense_results.csv", index=False)
print("Defender results saved -> models/malware_defense_results.csv")


Defender results saved -> models/malware_defense_results.csv


In [10]:
# Defender: decide action based on prediction probability
def defender_action(prob, label, block_threshold=0.8):
    if label == 1 and prob >= block_threshold:
        return "block_ip"
    return "allow"

# Predicted probabilities
probs = xgb_model.predict_proba(X_test_scaled)
pred_label = y_pred
pred_prob  = probs.max(axis=1)

# Action log
defender_action_log = pd.DataFrame({
    "ip": range(len(pred_label)),   # or some identifier column if exists
    "predicted_label": pred_label,
    "prob": pred_prob,
    "defense_action": [defender_action(p,l) for p,l in zip(pred_prob, pred_label)]
})

# Save action log
defender_action_log.to_csv("models/defender_action_log.csv", index=False)
print("Defender results saved -> models/defender_action_log.csv")


Defender results saved -> models/defender_action_log.csv


In [11]:
# Blocklist: all IPs marked as block_ip
blocklist = defender_action_log[defender_action_log['defense_action'] == "block_ip"]
blocklist.to_csv("models/malware_blocklist.csv", index=False)
print("Blocked IPs saved -> models/malware_blocklist.csv")

# Quick stats
print("Total blocked IPs:", len(blocklist))
print(blocklist.head(10))


Blocked IPs saved -> models/malware_blocklist.csv
Total blocked IPs: 12000
   ip  predicted_label      prob defense_action
0   0                1  0.999976       block_ip
1   1                1  0.999976       block_ip
2   2                1  0.999976       block_ip
3   3                1  0.999976       block_ip
4   4                1  0.999976       block_ip
5   5                1  0.999976       block_ip
6   6                1  0.999976       block_ip
7   7                1  0.999976       block_ip
8   8                1  0.999976       block_ip
9   9                1  0.999973       block_ip


In [12]:
import shutil

os.makedirs("backup_models", exist_ok=True)
artifacts = [
    "models/malware_xgb.json",
    "models/malware_scaler.joblib",
    "models/malware_label_encoders.joblib",
    "models/malware_defense_results.csv",
    "models/defender_action_log.csv",
    "models/malware_blocklist.csv"
]

for f in artifacts:
    shutil.copy(f, "backup_models/")
print("Backup done -> backup_models/")


Backup done -> backup_models/


In [1]:
import os
import joblib
import xgboost as xgb
from xgboost import XGBClassifier
import pandas as pd
import numpy as np
import shap
from sklearn.feature_extraction.text import TfidfVectorizer

# -------------------------------
# 1️⃣ Set threat
# -------------------------------
THREAT_NAME = "malware"  

# -------------------------------
# 2️⃣ Paths
# -------------------------------
model_json = f"models/{THREAT_NAME}_xgb.json"
pkl_model = f"models/{THREAT_NAME}_xgboost_model.pkl"
defense_csv = f"models/{THREAT_NAME}_defense_results.csv"
vectorizer_path = f"models/{THREAT_NAME}_tfidf_vectorizer.pkl"
explain_log = f"models/{THREAT_NAME}_explain_log.csv"

# -------------------------------
# 3️⃣ Convert JSON → PKL if not exists
# -------------------------------
if not os.path.exists(pkl_model) and os.path.exists(model_json):
    booster = xgb.Booster()
    booster.load_model(model_json)
    clf = XGBClassifier()
    clf._Booster = booster
    clf._le = None
    joblib.dump(clf, pkl_model)
    print(f"Saved PKL model -> {pkl_model}")
else:
    print(f"PKL model exists -> {pkl_model}")

# -------------------------------
# 4️⃣ Load model and defense log
# -------------------------------
model = joblib.load(pkl_model)
defense_results = pd.read_csv(defense_csv)
print("Columns in defense CSV:", defense_results.columns.tolist())

# -------------------------------
# 5️⃣ Determine text column
# -------------------------------
text_column = None
for col in ["text", "text_sample"]:
    if col in defense_results.columns:
        text_column = col
        break

if text_column:
    defense_results.rename(columns={text_column: "text"}, inplace=True)
else:
    print("⚠️ No text column found. SHAP will use IDs instead.")

# -------------------------------
# 6️⃣ Filter quarantined samples
# -------------------------------
quarantine_samples = defense_results[defense_results["defense_action"] == "quarantine"].copy()
print(f"Total quarantined samples: {len(quarantine_samples)}")

# -------------------------------
# 7️⃣ TF-IDF Vectorizer
# -------------------------------
if os.path.exists(vectorizer_path):
    vectorizer = joblib.load(vectorizer_path)
else:
    vectorizer = TfidfVectorizer(max_features=3000)
    if "text" in defense_results.columns:
        vectorizer.fit(defense_results["text"].astype(str).tolist())
        joblib.dump(vectorizer, vectorizer_path)
        print(f"Saved new TF-IDF vectorizer -> {vectorizer_path}")
    else:
        vectorizer = None

if len(quarantine_samples) > 0 and vectorizer:
    X_features = vectorizer.transform(quarantine_samples["text"].astype(str).tolist())
    feature_names = np.array(vectorizer.get_feature_names_out())
else:
    X_features = None
    feature_names = None

# -------------------------------
# 8️⃣ SHAP Explainability
# -------------------------------
explain_data = []

if X_features is not None:
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_features)

    for i in range(len(quarantine_samples)):
        shap_row = shap_values[i].flatten() if isinstance(shap_values, np.ndarray) else shap_values[i].toarray().flatten()
        top_indices = np.argsort(np.abs(shap_row))[-10:][::-1]
        top_feats = feature_names[top_indices]
        top_vals = shap_row[top_indices]
        top_pairs = [f"{f} ({round(v,3)})" for f,v in zip(top_feats, top_vals)]

        row_dict = {"index": quarantine_samples.index[i]}
        if "text" in quarantine_samples.columns:
            row_dict["text"] = quarantine_samples.iloc[i]["text"][:120] + ("..." if len(quarantine_samples.iloc[i]["text"]) > 120 else "")
        else:
            row_dict["sample_id"] = quarantine_samples.iloc[i].get("ip_or_id", quarantine_samples.index[i])

        row_dict["top_contributing_features"] = ", ".join(top_pairs)
        explain_data.append(row_dict)
else:
    print("⚠️ No quarantined samples or text data. Explain log will be empty.")

# -------------------------------
# 9️⃣ Save Explainability CSV
# -------------------------------
os.makedirs("models", exist_ok=True)
explain_df = pd.DataFrame(explain_data)
explain_df.to_csv(explain_log, index=False)
print(f"Explainability log saved -> {explain_log}")
print(f"✅ Completed: {THREAT_NAME}")


Saved PKL model -> models/malware_xgboost_model.pkl
Columns in defense CSV: ['predicted_label', 'prediction_prob', 'defense_action']
⚠️ No text column found. SHAP will use IDs instead.
Total quarantined samples: 0
⚠️ No quarantined samples or text data. Explain log will be empty.
Explainability log saved -> models/malware_explain_log.csv
✅ Completed: malware
